# Part 2 — Ensemble Methods: Random Forest & AdaBoost

Comparing a single Decision Tree against two ensemble methods — Random Forest (bagging) and AdaBoost (boosting) — on the same Cryotherapy dataset, looking at accuracy, F1-score, training time, and sensitivity to `n_estimators`.


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

## Loading and Preprocessing (same as Part 1)

In [13]:
df = pd.read_excel('Cryotherapy.xlsx') 

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   sex                  90 non-null     int64  
 1   age                  90 non-null     int64  
 2   Time                 90 non-null     float64
 3   Number_of_Warts      90 non-null     int64  
 4   Type                 90 non-null     int64  
 5   Area                 90 non-null     int64  
 6   Result_of_Treatment  90 non-null     int64  
dtypes: float64(1), int64(6)
memory usage: 5.0 KB


In [14]:
df = pd.get_dummies(df, columns=['Type'], drop_first=True, dtype=int)

X = df.drop(columns=['Result_of_Treatment'])
y = df['Result_of_Treatment']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [16]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train data shape: {X_train_scaled.shape}")
print(f"Test data shape: {X_test_scaled.shape}")

Train data shape: (72, 7)
Test data shape: (18, 7)


In [17]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42), 
    'Random Forest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

comparison_results = []

for model_name, model in models.items():
    start_time = time.time()
    
    model.fit(X_train_scaled, y_train)
    
    # duration
    end_time = time.time()
    training_time = end_time - start_time
    
    # predicting and evaluating
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    comparison_results.append({
        'Model': model_name,
        'Accuracy': acc,
        'F1-Score': f1,
        'Training Time (sec)': training_time
    })

# turning into DataFrame
comparison_df = pd.DataFrame(comparison_results)
print("Comparison of Models:")
display(comparison_df) 

Comparison of Models:


,Model,Accuracy,F1-Score,Training Time (sec)
0,Decision Tree,0.777778,0.777778,0.002736
1,Random Forest,0.888889,0.888889,0.093553
2,AdaBoost,0.888889,0.888889,0.224096


## Comparing Decision Tree, Random Forest, and AdaBoost

## Accuracy vs. `n_estimators`

### Ensemble Methods

The single Decision Tree's accuracy (77.78%) is clearly lower than the two ensemble
methods. That comes down to overfitting: a lone tree tends to grow deep enough to
memorize noise in the (small) training set, which hurts its ability to generalize.

Both Random Forest and AdaBoost reach 88.89% accuracy. By combining predictions from
many small trees, ensemble methods sharply reduce variance (error from being overly
sensitive to the training data) and produce smoother, more sensible decision boundaries.

The test set here is tiny — only 18 samples. 77.78% means the model got 14 out of 18
right ($\frac{14}{18}$); 88.89% means 16 out of 18 ($\frac{16}{18}$). Random Forest and
AdaBoost misclassify exactly the same 2 samples — both effectively hit the ceiling of
what's learnable from this amount of data.


### Training Time

**Decision Tree (fastest — 0.002s):** only one tree is built, so the computation is
trivial.

**Random Forest (moderate — 0.093s):** uses bagging — trees are built in parallel and
independently of each other. It's slower than a single tree simply because many trees
(e.g. 100) need to be built, but the parallel, independent nature keeps it relatively
fast.

**AdaBoost (slowest — 0.224s):** uses boosting — trees (in this case, decision stumps)
must be built sequentially. The second tree can't start until the first is done and its
errors have been reweighted. That lack of parallelism is what drives AdaBoost's training
time up.


### Accuracy vs. `n_estimators`

**Early saturation:** the Cryotherapy training set has only 72 samples. With as few as
10–30 trees, Random Forest and AdaBoost already have more than enough capacity to cover
every meaningful pattern and combination in that small amount of data.

Adding more trees (going up to 100 or 150) doesn't introduce new information or
diversity — the additional trees just vote on combinations that are largely redundant
with the earlier ones. That's why the accuracy curve flattens out almost immediately (or
after a brief initial jump) and stays flat from there.
